# Argument and research validation
Synthetic cards exercise the same parser, index builder, research context and query port used by the CLI.

In [ ]:
import sys, tempfile, shutil
from pathlib import Path
root = Path.cwd()
while not (root / "knowledge_palace").is_dir():
    root = root.parent
sys.path.insert(0, str(root))
from knowledge_palace.tests.test_research import ARGUMENT, SYNTHESIS, MINI
from knowledge_palace.graph.builder import write_index, ensure_index
from knowledge_palace.graph.port import GraphQueryPort
from knowledge_palace.interaction.research_helpers import research_context, render_progress
temp = tempfile.TemporaryDirectory()
vault, state = Path(temp.name)/"vault", Path(temp.name)/"state"
shutil.copytree(MINI, vault)
paper = vault/"papers/alpha-2020-echo.md"
paper.write_text(paper.read_text(encoding="utf-8") + ARGUMENT, encoding="utf-8")
brief = vault / "briefs/2026-07-13-gaps.md"
brief.write_text(brief.read_text(encoding="utf-8") + SYNTHESIS, encoding="utf-8")
_, document = write_index(vault, state)
document["payload"]["identity_report"]

In [ ]:
context = research_context(document["payload"], "rare-window-token")
context["works"], context["relations"]

In [ ]:
response = GraphQueryPort(vault, state).query_context("claim:alpha-2020-echo#C1")
assert response["node"]["id"] == "claim:alpha-2020-echo#C1"
assert context["relations"][0]["attrs"]["claim_ref"] == "claim:alpha-2020-echo#C1"
print(render_progress(context))


In [ ]:
context = research_context(document["payload"], "echo", ["rare-window-token"])
assert len(context["attempts"]) == 2
assert context["current_syntheses"]
assert len(context["candidates"]) <= 15
context["current_syntheses"], context["unread_dependencies"], context["bridges"]


## Bounded reading plan
`research ask` prints this view: every candidate, a few quoted Claims per work, query matches first.

In [ ]:
from knowledge_palace.interaction.research_helpers import bounded_context
view = bounded_context(research_context(document["payload"], "noise floors"), 1)
quoted = view["works"]["work:alpha-2020-echo"]
assert quoted["claims_total"] == 2 and [claim["id"] for claim in quoted["claims"]] == ["C2"]  # the Claim the query matched
assert view["relations"] == []  # C1's relation is left to the card, with C1
view["candidates"], quoted, view["gaps"]["gap:gap-echo-noise"]["statement"]

## Analysis views
`research source`, `bridges`, `novelty` and `partition` read the same payload and Vault identity.

In [ ]:
from knowledge_palace.graph.identity import domain_partition_report, load_vault_identity
from knowledge_palace.interaction.novelty import novelty_pipeline
from knowledge_palace.interaction.project_source import resolve
from knowledge_palace.interaction.research import reference_from_text
from knowledge_palace.semantic.corridors import find_corridors
identity = load_vault_identity(vault)
references = [reference_from_text(text) for text in ("arXiv:2101.00001v3", "https://doi.org/10.99999/Fixture.Alpha2020", "An Unread Paper")]
references, [resolve(reference, identity).get("work", "project-source") for reference in references]

In [ ]:
corridors = find_corridors(document["payload"], "alpha-domain", "beta-domain")
drill = novelty_pipeline(document["payload"], "echo drift")
corridors["total"], [entry["kind"] for entry in corridors["corridors"]], drill["corridors"]["total"], drill["works"]["total"]

In [ ]:
report = domain_partition_report(identity)
sorted(report["roots"]), dict(report["root_pairs"]), report["cards_without_root"]

In [ ]:
temp.cleanup()